# Omni-AD-30 全量运行 —— Swin 主干教程（448 高分辨率版）

工业图像异常检测（PatchCore + Swin Transformer）。本 notebook 从零跑通 30 类训练 / 预测 / 评测，输出官方指标表。

## 先看这里
- **需要 GPU 运行时**：右上角「代码执行程序 → 更改运行时类型 → T4 GPU」（有 A100 更快）。
- **数据要在 Google Drive**：`MyDrive/IAD/Omni-AD-30-release.zip`（打包后拖进 Drive，约 5.8GB）。
- **默认 448 高分辨率**（`features.3` 56×56 + `features.5` 28×28，像素定位比 224 细 4 倍）。
- 跑完记得把 `work/` 拷回 Drive 持久化，否则会话重置会丢。

## 0. 挂载 Drive + 检查 GPU/显存

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))
if torch.cuda.is_available():
    print("显存", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

## 1. 解压数据（已解压过可跳过；`-oq` 静默覆盖，重跑安全）

In [ ]:
%%bash
mkdir -p /content/data
unzip -oq /content/drive/MyDrive/IAD/Omni-AD-30-release.zip -d /content/data/
ls /content/data/Omni-AD-30-release | wc -l   # 期望 30

## 2. 干净克隆 swin 分支 + 装依赖

每次都 `rm -rf` 重新克隆，保证拿到最新代码（含 448 分辨率 + coreset 优化）。

In [ ]:
%%bash
cd /content
rm -rf IAD-Industrial-Anomaly-Detection
git clone -b swin https://github.com/coder-yu-WICK/IAD-Industrial-Anomaly-Detection.git
cd IAD-Industrial-Anomaly-Detection
git log --oneline -1
pip install -q onnx onnxscript onnxruntime-gpu scikit-learn

## 3. 软链数据 + 生成 manifest

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
mkdir -p data
ln -s /content/data/Omni-AD-30-release data/Omni-AD-30-release
python -u src/data/sample_manifest.py --data-root data/Omni-AD-30-release

## 4. 训练（默认 448，bank≈2 万 patch）

- 首次会联网下 `swin_t` 预训练权重（~110MB，之后缓存到 `model/pretrained/`）。
- 30 类预计 **1.5~3 小时**。
- **想更全的 bank**（更慢）：加 `--max-embed 300000`。
- **回退 224**（更快、定位稍粗）：把命令换成注释里那条。

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
python -u src/train.py \
  --data-root data/Omni-AD-30-release \
  --manifest data/Omni-AD-30-release/train_manifest.csv \
  --output-dir work/model_swin448 \
  --device cuda:0 --seed 2026 --num-workers 4

# ==== 回退 224：删掉上面 python 那 5 行，改用下面这条 ====
# python -u src/train.py \
#   --data-root data/Omni-AD-30-release \
#   --manifest data/Omni-AD-30-release/train_manifest.csv \
#   --output-dir work/model_swin224 \
#   --device cuda:0 --seed 2026 --num-workers 4 \
#   --input-size 256 256 --crop-size 224 224

## 5. 预测（自动识别主干/分辨率，从 ckpt 读，无需指定）

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
python -u src/predict.py \
  --data-root data/Omni-AD-30-release \
  --manifest data/Omni-AD-30-release/test_manifest.csv \
  --model-dir work/model_swin448 \
  --output-dir work/pred_swin448 \
  --device cuda:0 --num-workers 4

## 6. 评测 → 30 类指标表

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
python src/evaluate.py \
  --predictions-dir work/pred_swin448 \
  --data-root data/Omni-AD-30-release \
  --manifest data/Omni-AD-30-release/test_manifest.csv

## 附：跑完把产物拷回 Drive（防会话重置丢失）

```bash
cp -r work /content/drive/MyDrive/IAD/work_swin448
```

> 注意：`data/` 已 gitignore，数据严禁提交 GitHub；代码/commit 里不要出现学校信息。